# Certified edge zeros in a Dedekind eta scale family: reproduction notebook

This notebook recomputes every numerical statement of the manuscript that is not taken from the archived mpmath certificates.

| Section | Content | Rigour |
|---|---|---|
| 1 | Arb helpers, analytic constants | definitions |
| 2 | Laguerre and Rouché certificates at $a=23/16$; one Laguerre certificate at a parameter value inside each tracked excursion | rigorous (Arb balls) |
| 3 | Double zero at the right edge $a_R$: Banach/Krawczyk box, derivative signs | rigorous, local |
| 4 | Double zero at the left edge $a_L$: Newton, then the same box test | rigorous, local |
| 5 | Explicit checks at $a_c\pm10^{-4}$ | rigorous |
| 6 | Seven weight-one eta products; lattice-sum identity at $s=2$; LMFDB zeros at $a_0=\tfrac12\log3$ | finite checks; rigorous sign checks |
| 7 | Zero landscape for $0\le a\le1.75$, $0<x<20$ | exploratory float64, not rigorous |
| 8 | Save results, versions and figures; zip and download | |

**What is and is not proved.** The two box tests certify two double zeros with opposite local transitions: at $a_L$ two real zeros meet and leave the real axis, at $a_R$ a conjugate pair returns to it. They do not prove that the pair leaving at $a_L$ is the pair returning at $a_R$; that identification rests on the float64 tracking of Section 7. A negative Laguerre value proves that nonreal zeros exist at that parameter value but does not locate them.

**Runtime:** a CPU runtime is sufficient (no GPU needed); the whole notebook runs in about one to two minutes.

Analytic constants used below (derived in the manuscript): the tail of every integrand beyond $u=6$ is below $10^{-80}$; $|H_{zz}|<20$ for $|\mathrm{Im}\,z|<1$; $|H|<24$ on complex polydisks of radius $1/4$ about real points, hence $|\partial_a^3H|\le 9216$, $|\partial_a^3\partial_xH|\le 36864$ and $\|J(y)-J(b)\|_\infty\le 4\cdot 9216\,r_b<50000\,r_b$ on a box of radius $r_b$.

In [ ]:
%pip install -q "python-flint>=0.7"

In [ ]:
import json, time, platform, sys, datetime
from decimal import Decimal, ROUND_CEILING, ROUND_FLOOR
from fractions import Fraction
import flint
from flint import acb, arb, ctx, fmpq

ctx.prec = 320                      # about 96 decimal digits
I = acb(0, 1)
TAIL = arb(fmpq(1, 10**80))         # bound for the tail of every integral beyond u = 6
TOL = arb(2) ** -300
RESULTS, LOG = {}, []

def log(*args):
    s = " ".join(str(a) for a in args)
    print(s); LOG.append(s)

def exact(text):
    q = Fraction(text)
    return arb(fmpq(q.numerator, q.denominator))

def _dec(x):
    return Decimal(arb(x).mid().str(40, radius=False))

def up(x, d=4):
    # decimal string that is >= every point of the ball x (d significant digits, rounded up)
    u = _dec(arb(x).upper()); u = u + abs(u) * Decimal("1e-30")
    return format(u.quantize(Decimal(1).scaleb(u.adjusted() - d + 1), rounding=ROUND_CEILING), "E")

def lo(x, d=4):
    # decimal string that is <= every point of the ball x
    l = _dec(arb(x).lower()); l = l - abs(l) * Decimal("1e-30")
    return format(l.quantize(Decimal(1).scaleb(l.adjusted() - d + 1), rounding=ROUND_FLOOR), "E")

def encl(x, d=7):
    return [lo(x, d), up(x, d)]

def integrand(a, z, j):
    # W_a(w) * [cos(zw), -w sin(zw), -w^2 cos(zw)] for j = 0, 1, 2
    ea = acb(a).exp(); eia = 1 / ea
    def f(w, analytic):
        ew = w.exp()
        t1, t2 = ew * ea, ew * eia
        if not (t1.real > 0 and t2.real > 0):   # eta arguments must stay in the upper half-plane
            return acb("nan")
        W = (w / 2).exp() * (I * t1).modular_eta() * (I * t2).modular_eta()
        zw = z * w
        return W * zw.cos() if j == 0 else (-w * W * zw.sin() if j == 1 else -w * w * W * zw.cos())
    return f

def H(a, z, j=0):
    # rigorous enclosure of d^j/dz^j H(a, z): Arb integral over [0, 6] plus the tail ball
    val = acb.integral(integrand(a, acb(z), j), 0, 6, rel_tol=TOL, abs_tol=TOL)
    return val + acb(arb(0, TAIL), arb(0, TAIL))

def F_and_J(a, x, h=arb(fmpq(1, 10**15))):
    # F = (H, H_x) and J = DF at (a, x); H_a and H_ax by central differences plus Cauchy remainder
    H0, H1, H2 = (H(a, x, j).real for j in range(3))
    ra, rax = h * h * 9216 / 6, h * h * 36864 / 6
    Ha = (H(a + h, x).real - H(a - h, x).real) / (2 * h) + arb(0, ra.mid() + ra.rad())
    Hax = (H(a + h, x, 1).real - H(a - h, x, 1).real) / (2 * h) + arb(0, rax.mid() + rax.rad())
    return (H0, H1), ((Ha, H1), (Hax, H2))

def krawczyk(A, X, M, r_exp):
    # Banach test for G(y) = y - M F(y) on the sup-norm box of radius 10^-r_exp about (A, X)
    (f0, f1), J = F_and_J(A, X)
    M = [[arb(v) for v in row] for row in M]
    delta = max(abs(M[i][0] * f0 + M[i][1] * f1).upper() for i in range(2))
    IMJ = [[(1 if i == k else 0) - (M[i][0] * J[0][k] + M[i][1] * J[1][k]) for k in range(2)] for i in range(2)]
    nIMJ = max((abs(r[0]) + abs(r[1])).upper() for r in IMJ)
    nM = max((abs(r[0]) + abs(r[1])).upper() for r in M)
    rb = arb(fmpq(1, 10**r_exp))
    q = nIMJ + nM * 50000 * rb
    ok = bool(q < 1) and bool(delta + q * rb < rb)
    wid = arb(0, (50000 * rb).upper())       # J varies by at most 50000 r_b on the box
    return dict(ok=ok, delta=delta, q=q.upper(), dist=(delta / (1 - q)).upper(),
                Ha=J[0][0] + wid, Hxx=J[1][1] + wid, margin=(rb - delta - q * rb).lower())

log("python-flint", flint.__version__, "| precision", ctx.prec, "bits")

## 2. Nonreal zeros at $a=23/16$

Laguerre expression $\Lambda=H_z^2-HH_{zz}$ at $(a,x)=(23/16,74/5)$, and Rouché disks of radius $10^{-12}$ (as in the archived certificate) and $10^{-19}$ about $c=14.67872714035986218218+0.31107587891741700810\,i$. On the circle $|z-c|=r$ the comparison is $|H(c)|+10r^2<r\,|H_z(c)|$, using $|H_{zz}|<20$.

In [ ]:
t0 = time.time()
a1, x1 = exact("23/16"), exact("74/5")
h0, h1, h2 = (H(a1, x1, j).real for j in range(3))
lag = h1 * h1 - h0 * h2
log("Laguerre (23/16, 74/5):", lag.str(12, radius=True), "| negative:", bool(lag < 0))
RESULTS["laguerre"] = dict(a="23/16", x="74/5", enclosure=encl(lag, 9), negative=bool(lag < 0))

c = acb(exact("14.67872714035986218218"), exact("0.31107587891741700810"))
Hc, Hpc = H(a1, c, 0), H(a1, c, 1)
RESULTS["rouche_23_16"] = dict(center="14.67872714035986218218+0.31107587891741700810i", disks={})
for r_exp in (12, 19):
    r = arb(fmpq(1, 10**r_exp))
    lhs, rhs = abs(Hc) + 10 * r * r, r * abs(Hpc)
    ok = bool(lhs < rhs) and bool(c.imag > r)
    log(f"Rouche r=1e-{r_exp}: |H(c)|+10r^2 <= {up(lhs)}  <  r|H_z(c)| >= {lo(rhs)} : {ok}")
    RESULTS["rouche_23_16"]["disks"][f"1e-{r_exp}"] = dict(lhs_upper=up(lhs), rhs_lower=lo(rhs), ok=ok)
log(f"{time.time() - t0:.1f} s")

### 2b. One Laguerre certificate inside each tracked excursion

For each cluster of defects found by the exploratory scan of Section 7 (labelled A to G there), one representative point $(a,x)$ was fixed from that scan. A negative enclosure of $\Lambda(a,x)=H_x^2-HH_{xx}$ proves that $H(a,\cdot)$ has nonreal zeros. It does not locate them, so it does not show that they are the pair tracked in Section 7.

In [ ]:
t0 = time.time()
REPS = {"A": ("23/16", "74/5"), "B": ("1.2525", "12.387"), "C": ("1.5", "16.173"), "D": ("0.1775", "18.504"),
        "E": ("0.525", "19.124"), "F": ("0.7325", "18.802"), "G": ("1.0345", "20.046")}
RESULTS["laguerre_reps"] = {}
for name, (ta, tx) in REPS.items():
    aa, xx = exact(ta), exact(tx)
    g0, g1, g2 = (H(aa, xx, j).real for j in range(3))
    lam = g1 * g1 - g0 * g2
    log(f"{name}: Lambda({ta}, {tx}) in {encl(lam, 5)}  negative: {bool(lam < 0)}")
    RESULTS["laguerre_reps"][name] = dict(a=ta, x=tx, enclosure=encl(lam, 5), negative=bool(lam < 0))
log(f"{time.time() - t0:.1f} s")

## 3. Double zero at the right edge $a_R$

Box of radius $r_b$ about the archived rational centre $(a_c,x_c)$, with the archived integer preconditioner $M$. "dist" is the Banach bound $\delta/(1-q)$ for the distance from the centre to the unique solution of $H=H_x=0$. The signs $H_a<0<H_{xx}$ on the box mean that, locally, a conjugate pair of nonreal zeros returns to the real axis as $a$ increases through $a_R$.

In [ ]:
t0 = time.time()
A_STAR = exact("1.45200626937306544414442672465215400624761")
X_STAR = exact("14.8380401130710086477956510630327906219061")
M_STAR = [[-387250867, 0], [-948213121, 1602446079]]
RESULTS["right_edge"] = dict(center=["1.45200626937306544414442672465215400624761",
                                     "14.8380401130710086477956510630327906219061"], M=M_STAR, boxes={})
for r_exp in (19, 35):
    k = krawczyk(A_STAR, X_STAR, M_STAR, r_exp)
    sign_ok = bool(k["Ha"] < 0) and bool(k["Hxx"] > 0)
    log(f"a_R box 1e-{r_exp}: contraction {k['ok']} | delta <= {up(k['delta'])} | q <= {up(k['q'])} | "
        f"dist <= {up(k['dist'])} | H_a in {encl(k['Ha'])} | H_xx in {encl(k['Hxx'])} | H_a<0<H_xx {sign_ok}")
    RESULTS["right_edge"]["boxes"][f"1e-{r_exp}"] = dict(contraction=k["ok"], delta_upper=up(k["delta"]),
        q_upper=up(k["q"]), dist_upper=up(k["dist"]), H_a=encl(k["Ha"]), H_xx=encl(k["Hxx"]), signs_ok=sign_ok)
log(f"{time.time() - t0:.1f} s")

## 4. Double zero at the left edge $a_L$

Tracking the zero certified at $a=23/16$ backwards in $a$ in float64 (Section 7) suggests that it reaches the real axis near $(a,x)\approx(1.3485,13.88)$. Newton's method in midpoint arithmetic gives a centre; the rounded inverse Jacobian gives an integer preconditioner; then the box test of Section 3 is applied. The signs come out as $H_a>0$ and $H_{xx}>0$, the opposite of the pattern at $a_R$: locally, two real zeros meet and leave the real axis as $a$ increases through $a_L$. The test does not show that this pair is the one that returns at $a_R$.

In [ ]:
t0 = time.time()
a, x = arb("1.3485"), arb("13.8815")
for it in range(12):
    (f0, f1), ((j00, j01), (j10, j11)) = F_and_J(a, x)
    det = j00 * j11 - j01 * j10
    da, dx = (j11 * f0 - j01 * f1) / det, (-j10 * f0 + j00 * f1) / det
    a, x = arb(a.mid() - da.mid()), arb(x.mid() - dx.mid())
    if max(abs(da.mid()), abs(dx.mid())) < arb(10) ** -60:
        break
AL_TXT, XL_TXT = a.mid().str(45, radius=False), x.mid().str(45, radius=False)
A_L, X_L = exact(AL_TXT), exact(XL_TXT)
(_, _), ((j00, j01), (j10, j11)) = F_and_J(A_L, X_L)
det = (j00 * j11 - j01 * j10).mid()
M_L = [[round(float(j11.mid() / det)), round(float(-j01.mid() / det))],
       [round(float(-j10.mid() / det)), round(float(j00.mid() / det))]]
k = krawczyk(A_L, X_L, M_L, 30)
sign_ok = bool(k["Ha"] > 0) and bool(k["Hxx"] > 0)
log("a_L centre:", AL_TXT); log("x_L centre:", XL_TXT); log("M_L =", M_L)
log(f"a_L box 1e-30: contraction {k['ok']} | delta <= {up(k['delta'])} | q <= {up(k['q'])} | dist <= {up(k['dist'])} | "
    f"H_a in {encl(k['Ha'])} | H_xx in {encl(k['Hxx'])} | H_a>0, H_xx>0 {sign_ok}")
RESULTS["left_edge"] = dict(center=[AL_TXT, XL_TXT], M=M_L, newton_iterations=it + 1, box="1e-30",
    contraction=k["ok"], delta_upper=up(k["delta"]), q_upper=up(k["q"]), dist_upper=up(k["dist"]),
    H_a=encl(k["Ha"]), H_xx=encl(k["Hxx"]), signs_ok=sign_ok)
log(f"{time.time() - t0:.1f} s")

## 5. Explicit checks at $a_c\pm10^{-4}$

* $a=a_c+10^{-4}$: if $H$ takes the signs $+,-,+$ at three points near $x_c$, the intermediate value theorem gives at least two real zeros.
* $a=a_c-10^{-4}$: a Rouché disk about a Newton approximation of a nonreal zero.

These are checks at two parameter values; they do not by themselves give a uniform statement for every $a$ in between.

In [ ]:
t0 = time.time()
eps = arb(fmpq(1, 10**4))
a_up = A_STAR + eps
offs = ["-0.06", "0.000245", "0.06"]
vals = [H(a_up, X_STAR + arb(o)).real for o in offs]
signs = ["+" if v > 0 else ("-" if v < 0 else "?") for v in vals]
log("a_c+1e-4: signs", signs, "values", [v.str(5, radius=True) for v in vals])
a_dn = A_STAR - eps
z = acb(X_STAR + arb("0.000245"), arb("0.0288"))
for it in range(40):
    hz, dhz = H(a_dn, z, 0), H(a_dn, z, 1)
    step = hz / dhz
    z = acb(z.real.mid() - step.real.mid(), z.imag.mid() - step.imag.mid())
    if abs(step.mid()) < arb(10) ** -60:
        break
r = arb(fmpq(1, 10**13))
Hz, Hpz = H(a_dn, z, 0), H(a_dn, z, 1)
ok = bool(abs(Hz) + 10 * r * r < r * abs(Hpz)) and bool(z.imag > r)
zs = (z.real.mid().str(16, radius=False), z.imag.mid().str(16, radius=False))
log("a_c-1e-4: zero near", zs[0], "+", zs[1], "i ; Rouche r=1e-13:", ok)
RESULTS["near_right_edge"] = dict(eps="1e-4", plus=dict(x_offsets=offs, signs=signs, two_real_zeros=signs == ["+", "-", "+"]),
                                  minus=dict(zero=zs, rouche_radius="1e-13", ok=ok))
log(f"{time.time() - t0:.1f} s")

## 6. The seven weight-one eta products and the lattice identity

For $A+B=24$ and $a=\tfrac12\log(B/A)$ one has $e^ak^2+e^{-a}l^2=(Bk^2+Al^2)/\sqrt{AB}$, so the slice $H(a,\cdot)$ is the completed $L$-function of $f_{A,B}(\tau)=\eta(A\tau)\eta(B\tau)$. The first cell checks multiplicativity of the coefficients of $f_{A,B}$ for $n\le3000$ (a finite check of the known classification). The second compares $2H(a,z)$ at $s=\tfrac12+iz=2$ with the lattice sum $\Gamma(s)(12/\pi)^s\sum\chi_{12}(k)\chi_{12}(l)(e^ak^2+e^{-a}l^2)^{-s}$ truncated at $k,l\le K$.

In [ ]:
from math import gcd, isqrt, log as mlog
import numpy as np

def chi12(k):
    return {1: 1, 5: -1, 7: -1, 11: 1}.get(k % 12, 0)

def coeffs(A, B, nmax=3000):
    c = [0] * (nmax + 1)
    for k in range(1, isqrt(24 * nmax // A) + 2):
        if chi12(k) == 0 or A * k * k > 24 * nmax:
            continue
        for l in range(1, isqrt((24 * nmax - A * k * k) // B) + 2):
            e = A * k * k + B * l * l
            if chi12(l) and e % 24 == 0 and e // 24 <= nmax:
                c[e // 24] += chi12(k) * chi12(l)
    return c

def multiplicative(c, nmax=3000):
    return c[1] == 1 and all(c[m * n] == c[m] * c[n] for m in range(2, 60)
                             for n in range(m + 1, nmax // m + 1) if gcd(m, n) == 1)

RESULTS["eta_products"] = []
for A in range(1, 13):
    B = 24 - A
    mult = multiplicative(coeffs(A, B))
    log(f"eta({A:2d}t)eta({B:2d}t)  level {A*B:3d}  a = {0.5*mlog(B/A):.6f}  multiplicative up to 3000: {mult}")
    RESULTS["eta_products"].append(dict(A=A, B=B, level=A * B, a=round(0.5 * mlog(B / A), 6), multiplicative=mult))

# Gauss's criterion for eta(6t)eta(18t): for primes p > 3, c(p) = 2 if p = x^2 + 27 y^2,
# c(p) = -1 for the other p = 1 (mod 3), and c(p) = 0 for p = 2 (mod 3)
c618 = coeffs(6, 18)
def is_x2_27y2(p):
    return any(isqrt(p - 27 * y * y) ** 2 == p - 27 * y * y for y in range(1, isqrt(p // 27) + 1))
primes = [p for p in range(5, 3001) if all(p % d for d in range(2, isqrt(p) + 1))]
bad = [p for p in primes if c618[p] != (0 if p % 3 == 2 else (2 if is_x2_27y2(p) else -1))]
log(f"eta(6t)eta(18t): Gauss criterion checked for {len(primes)} primes 5 <= p <= 3000; mismatches: {bad}")
RESULTS["gauss_criterion_6_18"] = dict(primes_checked=len(primes), p_max=3000, mismatches=bad)

In [ ]:
K = 4000
kk = np.arange(1, K + 1); ch = np.array([chi12(int(v)) for v in kk], dtype=float)
nz = ch != 0; kk, ch = kk[nz].astype(float), ch[nz]
s = 2.0
RESULTS["lattice_check_s2"] = []
for A in (1, 2, 3, 4, 6, 8, 12):
    B = 24 - A; a = 0.5 * mlog(B / A)
    Q = np.exp(a) * kk[:, None] ** 2 + np.exp(-a) * kk[None, :] ** 2
    lat = float(np.sum(ch[:, None] * ch[None, :] * Q ** (-s))) * (12 / np.pi) ** s   # Gamma(2) = 1
    twoH = 2 * H(arb(a), acb(0, -1.5)).real                                          # s = 1/2 + iz = 2
    rel = abs(float(twoH.mid()) - lat) / abs(lat)
    log(f"A={A:2d}: 2H(a,-1.5i) = {twoH.str(12, radius=False)}  lattice(K={K}) = {lat:.12e}  rel. diff = {rel:.1e}")
    RESULTS["lattice_check_s2"].append(dict(A=A, twoH=twoH.str(15, radius=False), lattice=f"{lat:.15e}", rel_diff=f"{rel:.1e}"))

### 6b. The Artin point $a_0=\tfrac12\log3$ and the LMFDB zeros of 108.1.c.a

At $a_0$ the real zeros of $H(a_0,\cdot)$ are the ordinates of the zeros of $L(s,f)$, $f=\eta(6\tau)\eta(18\tau)$, on the critical line. For each of the ten ordinates $\gamma$ listed by the LMFDB for the $L$-function 2-108-3.2-c0-0-0, the cell evaluates $H(a_0,\gamma\pm10^{-25})$ in Arb. Opposite signs prove a real zero of $H(a_0,\cdot)$ within $10^{-25}$ of $\gamma$. This is a check of zero locations; it does not test the normalizing factor, which has no zeros and is checked at $s=2$ in Section 6 above.

In [ ]:
t0 = time.time()
LMFDB = ["2.82161410923512167788607608967", "4.54012115448883134462155601253", "6.14349374284521055741894837740",
         "7.20655230074699554971488628053", "8.645382261510086690609111542994", "9.764363290770792093036581350306",
         "10.66337417717924861137567235324", "12.11393391520275622911136162744", "12.81147910151262800003561171965",
         "13.92836271074950410633497392280"]
a0 = arb(3).log() / 2
d25 = arb(fmpq(1, 10**25))
RESULTS["artin_point_lmfdb"] = []
for gtxt in LMFDB:
    g = exact(gtxt)
    hl, hr = H(a0, g - d25).real, H(a0, g + d25).real
    ok = (bool(hl > 0) and bool(hr < 0)) or (bool(hl < 0) and bool(hr > 0))
    log(f"gamma = {gtxt}: H(a0, gamma -/+ 1e-25) signs {'+' if hl > 0 else '-'}{'+' if hr > 0 else '-'} -> zero within 1e-25: {ok}")
    RESULTS["artin_point_lmfdb"].append(dict(gamma=gtxt, zero_within_1e_25=ok))
log(f"{time.time() - t0:.1f} s")

## 7. Zero landscape (exploratory, float64)

Composite Gauss–Legendre quadrature (48 panels of 32 nodes on $[0,6]$). Real zeros are sign changes on a grid with step $0.001$ in $x$; a "defect" is a local minimum with $H>0$ or a local maximum with $H<0$, located by sign changes of the analytic derivative $H_x$ on the same grid, which signals a nearby nonreal pair. Nonreal pairs are then followed in $a$ by complex Newton steps until $|\mathrm{Im}\,z|<0.02$. Nothing in this section is rigorous.

In [ ]:
import matplotlib.pyplot as plt

g, gw = np.polynomial.legendre.leggauss(32)
edges = np.linspace(0.0, 6.0, 49)
U = np.concatenate([(e0 + e1) / 2 + (e1 - e0) / 2 * g for e0, e1 in zip(edges[:-1], edges[1:])])
WT = np.concatenate([(e1 - e0) / 2 * gw for e0, e1 in zip(edges[:-1], edges[1:])])

def eta_axis(v):
    y = np.exp(np.abs(v)); q = np.exp(-2 * np.pi * y)
    prod = np.ones_like(y)
    for n in range(1, 10):
        prod *= 1 - q ** n
    val = np.exp(-np.pi * y / 12) * prod
    return np.where(v < 0, val * np.exp(-v / 2), val)

def wcoef(a):
    return WT * np.exp(U / 2) * eta_axis(U + a) * eta_axis(U - a)

t0 = time.time()
AV = np.round(np.arange(0.0, 1.75001, 0.0025), 6)
XS = np.arange(0.0005, 20.0, 0.001)
Wm = np.stack([wcoef(a) for a in AV], axis=1)
Hm = np.empty((len(XS), len(AV))); Dm = np.empty((len(XS), len(AV)))
for i0 in range(0, len(XS), 2500):
    ph = np.outer(XS[i0:i0 + 2500], U)
    Hm[i0:i0 + 2500] = np.cos(ph) @ Wm
    Dm[i0:i0 + 2500] = -(np.sin(ph) * U) @ Wm          # analytic x-derivative (no differencing noise)
real_pts, defects = [], []
for j, a in enumerate(AV):
    h, dh = Hm[:, j], Dm[:, j]
    sc = np.nonzero(np.sign(h[:-1]) != np.sign(h[1:]))[0]
    real_pts += [(a, v) for v in XS[sc] - h[sc] * (XS[sc + 1] - XS[sc]) / (h[sc + 1] - h[sc])]
    ex = np.nonzero(np.sign(dh[:-1]) != np.sign(dh[1:]))[0]
    defects += [(a, XS[i]) for i in ex if (dh[i] < 0 < dh[i + 1] and h[i] > 0) or (dh[i] > 0 > dh[i + 1] and h[i] < 0)]
real_pts, defects = np.array(real_pts), np.array(defects)
log(f"scan: {len(AV)} values of a in [0, 1.75], x step 0.001 on (0, 20): {len(real_pts)} real-zero points, {len(defects)} defect points")
RESULTS["scan"] = dict(a_range=[0, 1.75], a_step=0.0025, x_range=[0, 20], x_step=0.001, n_real=int(len(real_pts)),
                       n_defects=int(len(defects)))
# accuracy of the float64 evaluator against Arb at 21 test points
errs = []
for a_t in (0.3, 1.0, 1.5):
    for x_t in (10.0, 14.0, 16.0, 17.0, 18.0, 19.0, 20.0):
        ref = float(H(arb(a_t), acb(x_t)).real.mid())
        errs.append(abs(float(np.cos(x_t * U) @ wcoef(a_t)) - ref))
log(f"float64 vs Arb at 21 points (a in {{0.3, 1, 1.5}}, 10 <= x <= 20): max abs error {max(errs):.1e}")
RESULTS["scan"]["float64_max_abs_error_vs_arb"] = f"{max(errs):.1e}"

def newton(a, z, it=40):
    cc = wcoef(a)
    for _ in range(it):
        hh, dh = np.cos(z * U) @ cc, -(np.sin(z * U) * U) @ cc
        z = z - hh / dh
        if abs(hh / dh) < 1e-13:
            break
    return z

def trace(a0, z0, da=0.0005):
    z0 = newton(a0, z0); out = [(a0, z0.real, z0.imag)]
    for sgn in (-1, 1):
        a, z = a0, z0
        while True:
            z2 = newton(a + sgn * da, z)
            if abs(z2.imag) < 0.02 or abs(z2 - z) > 0.2 or z2.real > 20.5:
                break
            a, z = a + sgn * da, z2
            out.append((a, z.real, z.imag))
    return np.array(sorted(out))

seeds = {"A": (23 / 16, 14.6787 + 0.3111j), "B": (1.27, 12.53 + 0.2j), "C": (1.51, 16.17 + 0.37j),
         "D": (0.1775, 18.504 + 0.05j), "E": (0.525, 19.124 + 0.03j), "F": (0.7325, 18.802 + 0.03j),
         "G": (1.0345, 20.046 + 0.03j)}
tracks = {n: trace(a, z) for n, (a, z) in seeds.items()}
# assign every defect point to the tracked excursion that passes next to it
owner = []
for a_d, x_d in defects:
    hit = [n for n, t in tracks.items()
           if np.any((np.abs(t[:, 0] - a_d) <= 0.0026) & (np.abs(t[:, 1] - x_d) < 0.35))]
    owner.append(hit[0] if hit else None)
counts = {n: owner.count(n) for n in tracks}
log("defect points per excursion:", counts, "| unassigned:", owner.count(None))
RESULTS["scan"]["defects_per_excursion"] = counts
RESULTS["scan"]["defects_unassigned"] = owner.count(None)
RESULTS["excursions"] = {}
for n, t in tracks.items():
    log(f"excursion {n}: a in [{t[0,0]:.4f}, {t[-1,0]:.4f}], Re z {t[0,1]:.2f} -> {t[-1,1]:.2f}, max Im z {t[:,2].max():.3f}")
    RESULTS["excursions"][n] = dict(a_first=round(float(t[0, 0]), 4), a_last=round(float(t[-1, 0]), 4),
        re_first=round(float(t[0, 1]), 2), re_last=round(float(t[-1, 1]), 2), max_im=round(float(t[:, 2].max()), 3))
log(f"{time.time() - t0:.1f} s")

In [ ]:
import matplotlib as mpl
# journal style: STIX serif (bundled with Matplotlib), inward ticks on all four sides, no grid
mpl.rcParams.update({
    "font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 10.5,
    "axes.linewidth": 0.8, "axes.edgecolor": "#1a1a1a", "axes.labelsize": 11.5,
    "xtick.direction": "in", "ytick.direction": "in", "xtick.top": True, "ytick.right": True,
    "xtick.minor.visible": True, "ytick.minor.visible": True,
    "xtick.major.size": 4.5, "ytick.major.size": 4.5, "xtick.minor.size": 2.2, "ytick.minor.size": 2.2,
    "legend.fancybox": False, "legend.edgecolor": "#1a1a1a", "legend.framealpha": 1.0, "legend.fontsize": 9,
})
PLUM, OCHRE = "#6b3fa0", "#c2850c"       # categorical pair (passes the CVD and contrast checks)
CONTEXT, INK = "#a3a3a3", "#1a1a1a"      # recessive context and text/rules
DASHDOT = (0, (7, 2.5, 1.5, 2.5))
ART = [(1, 23), (2, 22), (3, 21), (4, 20), (6, 18), (8, 16), (12, 12)]

fig, ax = plt.subplots(figsize=(9, 5.2))
ax.scatter(real_pts[:, 0], real_pts[:, 1], s=0.55, color=CONTEXT, linewidths=0, rasterized=True,
           label=r"real zeros of $H(a,\cdot)$")
if len(defects):
    ax.scatter(defects[:, 0], defects[:, 1], s=9, marker="^", color=PLUM, linewidths=0,
               label="defects (nearby nonreal pair)")
for A, B in ART:
    aa = 0.5 * np.log(B / A)
    ax.axvline(aa, color=INK, lw=0.7, ls=DASHDOT)
    ax.text(aa, 20.3, f"${A}\\cdot{B}$", ha="center", va="bottom", fontsize=9)
for n, t in tracks.items():
    i = int(np.argmax(t[:, 2]))
    xa, ya = t[i, 0], min(t[i, 1] + 0.45, 19.55)
    if t[i, 1] > 19.6:                 # cluster at the upper frame: label beside it
        xa, ya = xa + 0.035, 19.3
    ax.text(xa, ya, n, ha="center", fontsize=10.5,
            fontstyle="italic", fontweight="bold", color=INK)
ax.set_xlim(-0.02, 1.77); ax.set_ylim(0, 20); ax.set_xlabel(r"$a$"); ax.set_ylabel(r"$x$")
leg = ax.legend(loc="lower left", handletextpad=0.4)
hs = getattr(leg, "legend_handles", None) or getattr(leg, "legendHandles")
hs[0].set_sizes([12]); hs[1].set_sizes([30])
plt.tight_layout(); plt.savefig("fig_zero_landscape.png", dpi=300); plt.show()

aL, xL = float(AL_TXT), float(XL_TXT)
aS, xS = 1.452006269373065, 14.838040113071009
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6.6), sharex=True, gridspec_kw={"height_ratios": [2.3, 1]})
sel = (real_pts[:, 0] >= 1.15) & (real_pts[:, 0] <= 1.62) & (real_pts[:, 1] >= 10.5) & (real_pts[:, 1] <= 17.5)
for axx in (ax1, ax2):
    for aa in (0.5 * np.log(11), 0.5 * np.log(23)):
        axx.axvline(aa, color=INK, lw=0.8, ls=DASHDOT)
ax1.scatter(real_pts[sel, 0], real_pts[sel, 1], s=1.6, color=CONTEXT, linewidths=0, label="real zeros", rasterized=True)
for i, (n, t) in enumerate((k, tracks[k]) for k in ("B", "A", "C")):
    ax1.plot(t[:, 0], t[:, 1], color=PLUM, lw=1.7, label=r"nonreal pair, $\mathrm{Re}\,z$" if i == 0 else None)
    ax2.plot(t[:, 0], t[:, 2], color=PLUM, lw=1.7)
    ax1.text(t[len(t) // 2, 0], t[len(t) // 2, 1] + 0.35, n, ha="center", fontsize=11.5,
             fontstyle="italic", fontweight="bold", color=INK)
mk = dict(color=OCHRE, edgecolor=INK, linewidths=0.7, zorder=5)
ax1.scatter([23 / 16], [14.6787], marker="o", s=42, **mk, label=r"certified nonreal zero ($a=23/16$)")
ax1.scatter([aL, aS], [xL, xS], marker="*", s=150, **mk, label=r"certified double zeros $a_L$, $a_R$")
ax2.scatter([23 / 16], [0.3111], marker="o", s=42, **mk); ax2.scatter([aL, aS], [0, 0], marker="*", s=150, **mk)
ax1.text(0.5 * np.log(11), 17.62, r"$\eta(2\tau)\eta(22\tau)$", ha="center", fontsize=10)
ax1.text(0.5 * np.log(23), 17.62, r"$\eta(\tau)\eta(23\tau)$", ha="center", fontsize=10)
ax1.text(0.012, 0.975, "(a)", transform=ax1.transAxes, ha="left", va="top", fontsize=11)
ax2.text(0.012, 0.94, "(b)", transform=ax2.transAxes, ha="left", va="top", fontsize=11)
ax1.set_xlim(1.15, 1.62); ax1.set_ylim(10.5, 17.5); ax1.set_ylabel(r"$x=\mathrm{Re}\,z$")
ax1.legend(loc="lower right")
ax2.set_ylim(-0.03, 0.5); ax2.set_ylabel(r"$\mathrm{Im}\,z$"); ax2.set_xlabel(r"$a$")
plt.tight_layout(); plt.savefig("fig_excursions.png", dpi=300); plt.show()

## 8. Save, zip and download

Writes `results.json` (every number printed above), `run_log.txt`, `versions.json` and the two figures, packs them into `eta_edge_zeros_results.zip` and starts the download in Colab.

In [ ]:
import os, zipfile, importlib.metadata as md_
versions = dict(python=sys.version.split()[0], platform=platform.platform(),
                python_flint=flint.__version__, numpy=np.__version__,
                matplotlib=md_.version("matplotlib"), arb_precision_bits=int(ctx.prec),
                utc=datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S"))
try:
    versions["mpmath"] = md_.version("mpmath")
except Exception:
    versions["mpmath"] = "not installed"
with open("results.json", "w") as f: json.dump(RESULTS, f, indent=2)
with open("versions.json", "w") as f: json.dump(versions, f, indent=2)
with open("run_log.txt", "w") as f: f.write("\n".join(LOG) + "\n")
files_out = ["results.json", "versions.json", "run_log.txt", "fig_zero_landscape.png", "fig_excursions.png"]
with zipfile.ZipFile("eta_edge_zeros_results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in files_out:
        zf.write(fn)
print(json.dumps(versions, indent=2))
print("archive:", os.path.abspath("eta_edge_zeros_results.zip"))
try:
    from google.colab import files
    files.download("eta_edge_zeros_results.zip")
except ImportError:
    print("Not running in Colab: the archive is in the working directory.")